<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3 RBench Reproduction with Cosmos Framework

This notebook walks through generating the [RBench](https://huggingface.co/datasets/DAGroup-PKU/RBench) robotics video-generation benchmark with Cosmos3-Super using the native Cosmos Framework PyTorch entrypoint:

```bash
python -m cosmos_framework.scripts.inference
```

RBench is an **Image-to-Video (I2V)** benchmark: each case provides a single condition image and a prompt, and the model generates a short clip.

- **650 cases** split across 9 category files (e.g. `common_manipulation`, `dual_arm`, `humanoid`, ...).
- Each case is conditioned on one image (`imgs/<image_path>` in the dataset) plus a detailed `json_upsampled_prompt` and a shared `negative_prompt`.
- Generation target: **120 generated frames at 24 FPS** (we run `num_frames=121`, where frame 0 is the conditioning image).

We walk through one demo case (`common_manipulation_0001`) end-to-end. The full 650-case run is one cell at the end.

## Prerequisites

- Linux machine with NVIDIA GPU access (default recipe uses 4 GPUs).
- Model access on Hugging Face. Either run `uvx hf@latest auth login` or set `HF_TOKEN` in the environment.
- `uv >= 0.11.3` installed (https://docs.astral.sh/uv/getting-started/installation/).
- `git-lfs` on PATH. The RBench dataset (~22 GB) is downloaded by cloning the Hugging Face dataset repo, which stores images and checkpoints with Git LFS.
- Cache/output paths with enough disk space.

> **Headless servers:** if you see `libxcb.so.1: cannot open shared object file` when importing the model, install the system graphics libraries:
> ```bash
> apt-get install -y libxcb1 libgl1 libglib2.0-0
> ```

## 1. Configure Paths and Environment

All paths default to sensible locations under this `cosmos` checkout. Override any of them by exporting before launching the notebook:

```bash
export COSMOS3_REPO=/path/to/cosmos-framework
export COSMOS3_UV_GROUP=cu130-train   # or cu128-train
export UV_PROJECT_ENVIRONMENT=/path/to/large/uv/venvs/cosmos3-rbench
export COSMOS3_NUM_GPUS=4
export HF_HOME=/path/to/large/huggingface/cache
export CUDA_VISIBLE_DEVICES=0,1,2,3
export RBENCH_DATASET_ROOT=/path/to/RBench
export RBENCH_OUTPUT_ROOT=/path/to/rbench/outputs
```

In [ ]:
from pathlib import Path
import os
import socket


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "cookbooks").exists():
            return path
    return start


def free_local_port() -> str:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return str(sock.getsockname()[1])


def default_framework_repo(root: Path) -> Path:
    for candidate in (root / "packages" / "cosmos-framework", root / "packages" / "cosmos3"):
        if (candidate / "pyproject.toml").exists() and (candidate / "cosmos_framework").exists():
            return candidate
    return root / "packages" / "cosmos-framework"


COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
COSMOS3_REPO = Path(os.environ.get("COSMOS3_REPO", default_framework_repo(COSMOS_ROOT))).resolve()
COSMOS3_GIT_URL = os.environ.get("COSMOS3_GIT_URL", "git@github.com:NVIDIA/cosmos-framework.git")
COSMOS3_UV_GROUP = os.environ.get("COSMOS3_UV_GROUP", "cu130-train")
COSMOS3_UV_ENV = Path(os.environ.get("UV_PROJECT_ENVIRONMENT", COSMOS3_REPO / ".venv")).resolve()
COSMOS3_NUM_GPUS = os.environ.get("COSMOS3_NUM_GPUS", "4")
CUDA_VISIBLE_DEVICES = os.environ.get("CUDA_VISIBLE_DEVICES", "0,1,2,3")

RBENCH_NOTEBOOK_ROOT = COSMOS_ROOT / "evaluation" / "rbench"
RBENCH_ASSETS = RBENCH_NOTEBOOK_ROOT / "assets"
RBENCH_PROMPTS_DIR = RBENCH_ASSETS / "prompts"
RBENCH_HF_URL = os.environ.get("RBENCH_HF_URL", "https://huggingface.co/datasets/DAGroup-PKU/RBench")
RBENCH_DATASET_ROOT = Path(
    os.environ.get("RBENCH_DATASET_ROOT", RBENCH_NOTEBOOK_ROOT / "RBench")
).resolve()
RBENCH_IMGS_DIR = RBENCH_DATASET_ROOT / "imgs"
RBENCH_OUTPUT_ROOT = Path(
    os.environ.get("RBENCH_OUTPUT_ROOT", RBENCH_NOTEBOOK_ROOT / "outputs")
).resolve()
RBENCH_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DEMO_CASE_ID = os.environ.get("RBENCH_DEMO_CASE", "common_manipulation_0001")
CHECKPOINT = os.environ.get("RBENCH_CHECKPOINT", "Cosmos3-Super")

MASTER_ADDR = os.environ.get("COSMOS3_MASTER_ADDR", "127.0.0.1")
MASTER_PORT_I2V = os.environ.get("COSMOS3_I2V_MASTER_PORT", free_local_port())

for key, value in [
    ("COSMOS_ROOT", COSMOS_ROOT),
    ("COSMOS3_REPO", COSMOS3_REPO),
    ("COSMOS3_UV_ENV", COSMOS3_UV_ENV),
    ("COSMOS3_NUM_GPUS", COSMOS3_NUM_GPUS),
    ("CUDA_VISIBLE_DEVICES", CUDA_VISIBLE_DEVICES),
    ("RBENCH_PROMPTS_DIR", RBENCH_PROMPTS_DIR),
    ("RBENCH_DATASET_ROOT", RBENCH_DATASET_ROOT),
    ("RBENCH_OUTPUT_ROOT", RBENCH_OUTPUT_ROOT),
    ("DEMO_CASE_ID", DEMO_CASE_ID),
    ("CHECKPOINT", CHECKPOINT),
]:
    print(f"{key}={value}")

# Export for the %%bash cells below.
os.environ["COSMOS3_REPO"] = str(COSMOS3_REPO)
os.environ["COSMOS3_GIT_URL"] = COSMOS3_GIT_URL
os.environ["COSMOS3_UV_GROUP"] = COSMOS3_UV_GROUP
os.environ["COSMOS3_UV_ENV"] = str(COSMOS3_UV_ENV)
os.environ["UV_PROJECT_ENVIRONMENT"] = str(COSMOS3_UV_ENV)
os.environ["COSMOS3_NUM_GPUS"] = COSMOS3_NUM_GPUS
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
os.environ["RBENCH_HF_URL"] = RBENCH_HF_URL
os.environ["RBENCH_DATASET_ROOT"] = str(RBENCH_DATASET_ROOT)
os.environ["RBENCH_OUTPUT_ROOT"] = str(RBENCH_OUTPUT_ROOT)
os.environ["DEMO_CASE_ID"] = DEMO_CASE_ID
os.environ["CHECKPOINT"] = CHECKPOINT
os.environ["COSMOS3_MASTER_ADDR"] = MASTER_ADDR
os.environ["COSMOS3_I2V_MASTER_PORT"] = MASTER_PORT_I2V

## 2. Clone or Reuse Cosmos Framework

In [ ]:
%%bash
set -euo pipefail

mkdir -p "$(dirname "$COSMOS3_REPO")"

if [ -f "$COSMOS3_REPO/pyproject.toml" ] && [ -d "$COSMOS3_REPO/cosmos_framework" ]; then
  echo "Using existing framework checkout: $COSMOS3_REPO"
elif [ -e "$COSMOS3_REPO" ]; then
  echo "COSMOS3_REPO exists but is not a Cosmos Framework checkout: $COSMOS3_REPO"
  exit 1
else
  echo "Cloning $COSMOS3_GIT_URL into $COSMOS3_REPO"
  git clone "$COSMOS3_GIT_URL" "$COSMOS3_REPO"
fi

cd "$COSMOS3_REPO"
git status --short --branch
git remote -v

## 3. Install Native PyTorch Dependencies

Installs framework dependencies with the requested CUDA group (default `cu130-train`).

In [ ]:
%%bash
set -euo pipefail

if ! command -v uv >/dev/null 2>&1; then
  echo "uv is not installed. Install it first: https://docs.astral.sh/uv/getting-started/installation/"
  exit 1
fi

export GIT_LFS_SKIP_SMUDGE=1
cd "$COSMOS3_REPO"
export UV_PROJECT_ENVIRONMENT="${UV_PROJECT_ENVIRONMENT:-$COSMOS3_UV_ENV}"
echo "Using UV_PROJECT_ENVIRONMENT=$UV_PROJECT_ENVIRONMENT"
uv sync --all-extras --group="$COSMOS3_UV_GROUP"
if [ ! -x "$COSMOS3_UV_ENV/bin/python" ]; then
  echo "uv sync completed, but expected Python is missing: $COSMOS3_UV_ENV/bin/python"
  exit 1
fi

## 4. Verify GPU and Python Environment

In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
if [ ! -x "$COSMOS3_UV_ENV/bin/python" ]; then
  echo "Missing $COSMOS3_UV_ENV/bin/python"
  echo "Run the Install Native PyTorch Dependencies cell first."
  exit 1
fi
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" "$COSMOS3_UV_ENV/bin/python" - <<'PY'
import torch
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(f"device {index}:", torch.cuda.get_device_name(index))
PY

## 5. Download the RBench Dataset

RBench is hosted on Hugging Face at [`DAGroup-PKU/RBench`](https://huggingface.co/datasets/DAGroup-PKU/RBench). We download it by cloning the dataset repo with Git LFS.

Layout (under `$RBENCH_DATASET_ROOT`):

```
RBench/
├── assets/        # benchmark assets
├── checkpoints/   # BERT checkpoints (for the official RBench scorer; not used here)
├── imgs/          # 650 condition images, organized by category (imgs/<category>/<name>.jpg)
└── prompts/       # 9 source prompt files
```

> The positive/negative prompts used for generation come from the **local** assets at `assets/prompts/` (which include `json_upsampled_prompt` and `negative_prompt`). Only the condition images are read from the cloned dataset.

In [ ]:
%%bash
set -euo pipefail

if [ -d "$RBENCH_DATASET_ROOT/imgs" ]; then
  echo "RBench dataset already present at $RBENCH_DATASET_ROOT"
  ls "$RBENCH_DATASET_ROOT"
  exit 0
fi

if ! command -v git-lfs >/dev/null 2>&1; then
  echo "git-lfs is required to download the RBench dataset (LFS-backed images/checkpoints)." >&2
  echo "Install it: https://git-lfs.com/  (e.g. apt-get install -y git-lfs)" >&2
  exit 1
fi

git lfs install
mkdir -p "$(dirname "$RBENCH_DATASET_ROOT")"
git clone "$RBENCH_HF_URL" "$RBENCH_DATASET_ROOT"

echo "--- contents of $RBENCH_DATASET_ROOT ---"
ls "$RBENCH_DATASET_ROOT"

## 6. Load Prompts and Preview the Demo Case

We combine the 9 local category prompt files under `assets/prompts/` into a single `name -> entry` mapping (650 cases total). Each entry carries the `json_upsampled_prompt` (used as the positive prompt) and the `negative_prompt`. The condition image for a case is `imgs/<image_path>` inside the cloned dataset.

In [ ]:
import json
from IPython.display import Image, display

prompt_files = sorted(RBENCH_PROMPTS_DIR.glob("*.json"))
assert prompt_files, f"No prompt files found under {RBENCH_PROMPTS_DIR}"

PROMPTS: dict[str, dict] = {}
for path in prompt_files:
    for row in json.loads(path.read_text()):
        name = row["name"]
        if name in PROMPTS:
            raise ValueError(f"Duplicate prompt name across files: {name}")
        PROMPTS[name] = row

print(f"Loaded {len(PROMPTS)} prompts from {len(prompt_files)} files:")
for path in prompt_files:
    print(" ", path.name)
assert len(PROMPTS) == 650, f"expected 650 prompts, got {len(PROMPTS)}"

entry = PROMPTS[DEMO_CASE_ID]
demo_image = RBENCH_IMGS_DIR / entry["image_path"]
print("\ndemo case:", DEMO_CASE_ID)
print("condition image:", demo_image)
print("short prompt:", entry.get("prompt"))

if demo_image.exists():
    display(Image(filename=str(demo_image), width=480))
else:
    print("Condition image not found yet - run the dataset download cell first.")

## 7. Helper Functions

**I2V recipe**

- **Conditioning**: the single condition image. The model loads it as the first generated frame.
- **Output length**: 121 frames at 24 fps. Frame 0 is the conditioning image; frames 1–120 are the 5 seconds of generated content.
- **Sampling**: `num_steps=50`, `guidance=6.0`, `shift=10.0`, `seed=0` (all other sampler settings left at framework defaults).
- **Positive prompt**: the per-case `json_upsampled_prompt` string, passed through verbatim.
- **Negative prompt**: the shared `negative_prompt` string, passed through verbatim.

Helpers:

- `case_image_path(case_id)` — resolve the condition image for a case from the cloned dataset.
- `build_i2v_row(case_id)` — assemble the inference JSONL row for a case.
- `build_i2v_input_jsonl(case_id, dst)` — write a one-line JSONL for a single case.

In [ ]:
I2V_NUM_FRAMES = 121
I2V_FPS = 24
I2V_RESOLUTION = "720"
I2V_ASPECT_RATIO = "16,9"
I2V_NUM_STEPS = 50
I2V_GUIDANCE = 6.0
I2V_SHIFT = 10.0
I2V_SEED = 0


def case_image_path(case_id: str) -> Path:
    entry = PROMPTS[case_id]
    image_path = RBENCH_IMGS_DIR / entry["image_path"]
    if not image_path.exists():
        raise FileNotFoundError(f"condition image not found for {case_id}: {image_path}")
    return image_path


def build_i2v_row(case_id: str) -> dict:
    entry = PROMPTS[case_id]
    return {
        "aspect_ratio": I2V_ASPECT_RATIO,
        "fps": I2V_FPS,
        "guidance": I2V_GUIDANCE,
        "model_mode": "image2video",
        "name": case_id,
        "negative_prompt": entry["negative_prompt"],
        "num_frames": I2V_NUM_FRAMES,
        "num_outputs": 1,
        "num_steps": I2V_NUM_STEPS,
        "prompt": entry["json_upsampled_prompt"],
        "resolution": I2V_RESOLUTION,
        "seed": I2V_SEED,
        "shift": I2V_SHIFT,
        "vision_path": str(case_image_path(case_id)),
    }


def build_i2v_input_jsonl(case_id: str, dst_jsonl: Path) -> Path:
    dst_jsonl.parent.mkdir(parents=True, exist_ok=True)
    dst_jsonl.write_text(json.dumps(build_i2v_row(case_id)) + "\n")
    return dst_jsonl


def display_video(path: Path, width: int = 480) -> None:
    from IPython.display import Video, display
    display(Video(filename=str(path), embed=True, width=width))

## 8. I2V — Build the Input JSONL for the Demo Case

In [ ]:
i2v_run_dir = RBENCH_OUTPUT_ROOT / "i2v" / DEMO_CASE_ID
i2v_run_dir.mkdir(parents=True, exist_ok=True)

i2v_input_jsonl = i2v_run_dir / "input.jsonl"
i2v_output_dir = i2v_run_dir / "raw"
i2v_output_dir.mkdir(parents=True, exist_ok=True)

build_i2v_input_jsonl(DEMO_CASE_ID, i2v_input_jsonl)

os.environ["I2V_INPUT"] = str(i2v_input_jsonl)
os.environ["I2V_OUTPUT_DIR"] = str(i2v_output_dir)

print("I2V_INPUT =", i2v_input_jsonl)
print("I2V_OUTPUT_DIR =", i2v_output_dir)
print()
print("row preview:")
print(i2v_input_jsonl.read_text()[:600], "...")

### Run I2V Inference

We use the **latency** parallelism preset with context-parallel sharding across all visible GPUs (`--cp-size=$COSMOS3_NUM_GPUS`). For a single-case run on 4×H100, this completes in roughly 1–2 minutes.

In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_I2V_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --dp-shard-size=1 --dp-replicate-size=1 \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$I2V_INPUT" \
  -o "$I2V_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails

### Preview the Raw Output

The raw generation is a 121-frame mp4 at 24 fps (frame 0 is the conditioning image). We keep it as-is.

In [ ]:
raw_outputs = sorted(i2v_output_dir.rglob("vision.mp4"))
if raw_outputs:
    raw_mp4 = raw_outputs[0]
    print("raw output:", raw_mp4)
    display_video(raw_mp4)
else:
    print("No vision.mp4 found yet - run the inference cell first.")

## 9. (Optional) Run All 650 Cases

The cell below mirrors the demo flow but writes a single combined JSONL covering every case (all 9 categories), then the following bash cell generates all 650 I2V outputs in one run.

Run them only if you intend to generate the full benchmark.

In [ ]:
RUN_ALL = False  # set to True to enable the full 650-case sweep

if RUN_ALL:
    case_ids = sorted(PROMPTS.keys())
    assert len(case_ids) == 650

    all_inputs_dir = RBENCH_OUTPUT_ROOT / "i2v_full" / "inputs"
    all_raw_dir = RBENCH_OUTPUT_ROOT / "i2v_full" / "raw"
    all_inputs_dir.mkdir(parents=True, exist_ok=True)
    all_raw_dir.mkdir(parents=True, exist_ok=True)

    all_jsonl = all_inputs_dir / "all_650.jsonl"
    with all_jsonl.open("w") as fp:
        for case_id in case_ids:
            fp.write(json.dumps(build_i2v_row(case_id)) + "\n")

    os.environ["I2V_FULL_INPUT"] = str(all_jsonl)
    os.environ["I2V_FULL_OUTPUT_DIR"] = str(all_raw_dir)
    print("I2V_FULL_INPUT =", all_jsonl)
    print("I2V_FULL_OUTPUT_DIR =", all_raw_dir)
    print(f"Wrote {len(case_ids)} rows. Run the next bash cell to generate all 650 I2V outputs.")
else:
    print("Set RUN_ALL = True above and re-run to enable the full 650-case sweep.")

### Generate All 650 I2V Outputs

In [ ]:
%%bash
set -euo pipefail

if [ -z "${I2V_FULL_INPUT:-}" ]; then
  echo "Set RUN_ALL = True in the previous Python cell and re-run it first."
  exit 0
fi

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_I2V_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --dp-shard-size=1 --dp-replicate-size=1 \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$I2V_FULL_INPUT" \
  -o "$I2V_FULL_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails